In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder

In [3]:
# LOAD THE DATA
crime = pd.read_csv("NYPD_Arrest_Data_YTD.csv")
print("Shape:", crime.shape)
print("Columns:\n", crime.columns)
print("\nMissing values:\n", crime.isnull().sum())

Shape: (69305, 19)
Columns:
 Index(['ARREST_KEY', 'ARREST_DATE', 'PD_CD', 'PD_DESC', 'KY_CD', 'OFNS_DESC',
       'LAW_CODE', 'LAW_CAT_CD', 'ARREST_BORO', 'ARREST_PRECINCT',
       'JURISDICTION_CODE', 'AGE_GROUP', 'PERP_SEX', 'PERP_RACE', 'X_COORD_CD',
       'Y_COORD_CD', 'Latitude', 'Longitude', 'Location'],
      dtype='str')

Missing values:
 ARREST_KEY             0
ARREST_DATE            0
PD_CD                  2
PD_DESC                0
KY_CD                 12
OFNS_DESC              0
LAW_CODE               0
LAW_CAT_CD           267
ARREST_BORO            0
ARREST_PRECINCT        0
JURISDICTION_CODE      0
AGE_GROUP              0
PERP_SEX               0
PERP_RACE              0
X_COORD_CD             0
Y_COORD_CD             0
Latitude               0
Longitude              0
Location               0
dtype: int64


In [4]:
# BASIC CLEANING
# Convert arrest date to datetime
crime["ARREST_DATE"] = pd.to_datetime(
    crime["ARREST_DATE"])

# Extract time features
crime["year"] = crime["ARREST_DATE"].dt.year
crime["month"] = crime["ARREST_DATE"].dt.month
crime["day"] = crime["ARREST_DATE"].dt.day
crime["day_of_week"] = crime["ARREST_DATE"].dt.day_name()
crime["weekend"] = crime["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)

# Replace literal '(null)' strings with true NaN
crime = crime.replace("(null)", np.nan)

# Remove duplicate rows if any
crime = crime.drop_duplicates()

# Ensure precinct is numeric
crime["ARREST_PRECINCT"] = pd.to_numeric(
    crime["ARREST_PRECINCT"])

# Encode categorical variables
encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(crime[["LAW_CAT_CD", "ARREST_BORO"]])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(["LAW_CAT_CD", "ARREST_BORO"]))
crime = pd.concat([crime.reset_index(drop=True), encoded_df], axis=1)

print("\nCleaned Shape:", crime.shape)


Cleaned Shape: (69305, 35)


In [ ]:
# EXPLORATORY DATA ANALYSIS
# Histogram: arrests by precinct
plt.figure(figsize=(10, 6))
crime["ARREST_PRECINCT"].hist(bins=40)
plt.title("Distribution of Arrests by Precinct")
plt.xlabel("Precinct")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Proportional distribution of offense type
plt.figure(figsize=(12, 6))
crime["OFNS_DESC"].value_counts().head(15).plot(kind="bar")
plt.title("Top 15 Offense Types")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Distribution by day of week
order = ["Monday", "Tuesday", "Wednesday", "Thursday","Friday", "Saturday", "Sunday"]

plt.figure(figsize=(10, 6))
sns.countplot(data=crime, x="day_of_week", order=order)
plt.title("Crime Distribution by Day of Week")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Correlation heatmap
numeric_cols = ["ARREST_PRECINCT", "Latitude", "Longitude", "month", "day", "weekend"]

plt.figure(figsize=(8, 6))
sns.heatmap(crime[numeric_cols].corr(), annot=True, cmap="viridis", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [7]:
# Save to a new CSV for your team
crime.to_csv('NYPD_Data.csv', index=False)